# 02 Pipeline

Build a governed pipeline in seven steps: **Environment → Data Contracts → Read → Transform → Target → Validate → Write**.

## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.2.0 | Voyce | 6 Aug 2026 |

This redesigned version has local structural and public-API compatibility validation only. Run it in your configured Fabric workspace before recording a new runtime test.

# 0. Environment

Run the shared Fabric configuration and import the public APIs used by this pipeline.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    # FabricOps v0.1.0 onwards
    widget_view_catalogue,
    # FabricOps v0.2.0 onwards
    check_dq,
    check_freshness,
    check_schema,
    check_sensitive_data,
    check_source_stability,
    observe_table,
    pipeline_read,
    pipeline_write,
    profile_table,
    resolve_table_id,
    widget_select_data_contract,
)

catalogue_widget = widget_view_catalogue(mode="explore")

# 1. Data Contracts

Select the Data Contracts to test with this pipeline. Production automatically uses activated Data Contracts.

In [ ]:
CONTRACTS = widget_select_data_contract()

# 2. Read

Describe each governed source once. FabricOps resolves its canonical table identity and physical read path; engineering checks and profiling remain explicit.

## READ 1 — Orders

In [ ]:
READ_STORE = "source"
READ_SCHEMA = "demo"
READ_TABLE = "orders"
READ_QUERY = None

orders_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

orders_df = orders_result["dataframe"]
ORDERS_TABLE_ID = orders_result["table_id"]

check_schema(ORDERS_TABLE_ID, dataframe=orders_df, raise_on_failure=True)
check_dq(orders_df, table_id=ORDERS_TABLE_ID, raise_on_failure=True)
orders_profile = profile_table(table_id=ORDERS_TABLE_ID)
display(orders_profile["profile"])

catalogue_widget["show"](table_id=ORDERS_TABLE_ID)

## READ 2 — Products

In [ ]:
READ_STORE = "source"
READ_SCHEMA = "demo"
READ_TABLE = "products"
READ_QUERY = None

products_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

products_df = products_result["dataframe"]
PRODUCTS_TABLE_ID = products_result["table_id"]

check_schema(PRODUCTS_TABLE_ID, dataframe=products_df, raise_on_failure=True)
check_dq(products_df, table_id=PRODUCTS_TABLE_ID, raise_on_failure=True)
products_profile = profile_table(table_id=PRODUCTS_TABLE_ID)
display(products_profile["profile"])

catalogue_widget["show"](table_id=PRODUCTS_TABLE_ID)

## READ 3 — Order History

In [ ]:
READ_STORE = "product"
READ_SCHEMA = "demo"
READ_TABLE = "order_history"
READ_QUERY = f"""
SELECT
    customer_id,
    COUNT(*) AS historical_order_count,
    SUM(net_amount) AS historical_net_amount,
    MAX(order_datetime) AS latest_historical_order_datetime
FROM {READ_SCHEMA}.{READ_TABLE}
GROUP BY customer_id
"""

history_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

history_df = history_result["dataframe"]
HISTORY_TABLE_ID = history_result["table_id"]

# This query result is derived engineering data, not the complete governed physical table.
history_profile = profile_table(dataframe=history_df)
display(history_profile["profile"])

catalogue_widget["show"](table_id=HISTORY_TABLE_ID)

# 3. Transform

Business transformations remain project-owned PySpark.

In [ ]:
transformed_df = (
    orders_df.alias("orders")
    .join(products_df.alias("products"), on="product_id", how="left")
    .join(history_df.alias("history"), on="customer_id", how="left")
    .withColumn(
        "order_net_amount",
        F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2),
    )
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)
display(transformed_df)

# 4. Target

Describe the governed target and the processing strategy being proposed for this pipeline.

In [ ]:
WRITE_STORE = "unified"
WRITE_SCHEMA = "demo"
WRITE_TABLE = "curated_orders"
WRITE_LOAD_STRATEGY = "overwrite"

WRITE_TABLE_ID = resolve_table_id(
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
    table_name=WRITE_TABLE,
)

## Source observations

Freshness and Source Stability depend on the governed source-to-target relationship, so run them after defining the target.

In [ ]:
if orders_result["has_contract"]:
    orders_observation = observe_table(
        table_id=ORDERS_TABLE_ID,
        target_table_id=WRITE_TABLE_ID,
    )
    check_freshness(orders_observation, raise_on_failure=True)
    check_source_stability(orders_observation, target_table_id=WRITE_TABLE_ID)

if products_result["has_contract"]:
    products_observation = observe_table(
        table_id=PRODUCTS_TABLE_ID,
        target_table_id=WRITE_TABLE_ID,
    )
    check_freshness(products_observation, raise_on_failure=True)
    check_source_stability(products_observation, target_table_id=WRITE_TABLE_ID)

if history_result["has_contract"]:
    history_observation = observe_table(
        table_id=HISTORY_TABLE_ID,
        target_table_id=WRITE_TABLE_ID,
    )
    check_freshness(history_observation, raise_on_failure=True)
    check_source_stability(history_observation, target_table_id=WRITE_TABLE_ID)

# 5. Validate

Run target Guardrails explicitly before publication so failures remain visible to the engineer.

In [ ]:
check_schema(WRITE_TABLE_ID, dataframe=transformed_df, raise_on_failure=True)
check_dq(transformed_df, table_id=WRITE_TABLE_ID, raise_on_failure=True)

sensitive_result = check_sensitive_data(
    transformed_df,
    table_id=WRITE_TABLE_ID,
)
if not sensitive_result["can_continue"]:
    raise RuntimeError("A blocking Sensitive Data Guardrail failed.")

prepared_df = sensitive_result["dataframe"]

# 6. Write

FabricOps resolves the governed processing and physical publication path, then commits Lineage and Source Observation metadata only after the write succeeds.

## WRITE 1 — Curated Orders

The source table IDs declare the exact governed sources that contributed to this target so FabricOps can record accurate Lineage and Source Observation state after a successful write.

In [ ]:
write_result = pipeline_write(
    prepared_df,
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
    table_name=WRITE_TABLE,
    load_strategy=WRITE_LOAD_STRATEGY,
    source_table_ids=[
        ORDERS_TABLE_ID,
        PRODUCTS_TABLE_ID,
        HISTORY_TABLE_ID,
    ],
)

WRITE_TABLE_ID = write_result["table_id"]

write_profile = profile_table(table_id=WRITE_TABLE_ID)
display(write_profile["profile"])

catalogue_widget["show"](table_id=WRITE_TABLE_ID)